In [7]:
def to_mask(s):

    return sum(1 << (x - 1) for x in s)

def jaccard_index(a, b):

    union = a | b
    if not union:
        return 0
    
    return (a & b).bit_count() / union.bit_count()


In [8]:
universe = set(range(1, 11))

subsets = [
    {1, 2, 3},
    {2, 4, 5},
    {3, 5, 6, 7},
    {1, 7, 8},
    {8, 9},
    {9, 10},
    {4, 6, 10},
    {2, 3, 9},
]

universe_mask = to_mask(universe)
masks = [to_mask(s) for s in subsets]

In [9]:
# покрытие (маска) -> кортеж индексов подмножеств, которыми оно получено
combinations = {}

for i, m in enumerate(masks):
    if m not in combinations:
        combinations[m] = (i,)

seen = set(combinations)   # все покрытия, уже найденные на любом уровне

In [10]:
best_score = max(jaccard_index(c, universe_mask) for c in combinations)
best_comb = [c for c in combinations if jaccard_index(c, universe_mask) == best_score]

level = 1

while best_score < 1 and combinations:

    new_combinations = {}

    for cover, combination in combinations.items():

        for i, m in enumerate(masks):

            new_cover = cover | m

            # подмножество ничего нового не покрыло (в том числе оно уже в комбинации)
            if new_cover == cover:
                continue

            # такое покрытие уже получали: на этом уровне или раньше
            if new_cover in seen:
                continue

            seen.add(new_cover)
            new_combinations[new_cover] = combination + (i,)

    combinations = new_combinations

    if combinations:
        level += 1
        best_score = max(jaccard_index(c, universe_mask) for c in combinations)
        best_comb = [c for c in combinations
                     if jaccard_index(c, universe_mask) == best_score]

    print(f"Уровень: {level}, лучший score: {best_score}")
    for cover in best_comb:
        print([sorted(subsets[i]) for i in combinations[cover]])


Уровень: 2, лучший score: 0.6
[[1, 2, 3], [3, 5, 6, 7]]
[[1, 2, 3], [4, 6, 10]]
[[2, 4, 5], [3, 5, 6, 7]]
[[2, 4, 5], [1, 7, 8]]
[[3, 5, 6, 7], [1, 7, 8]]
[[3, 5, 6, 7], [8, 9]]
[[3, 5, 6, 7], [9, 10]]
[[3, 5, 6, 7], [4, 6, 10]]
[[3, 5, 6, 7], [2, 3, 9]]
[[1, 7, 8], [4, 6, 10]]
[[1, 7, 8], [2, 3, 9]]
[[4, 6, 10], [2, 3, 9]]
Уровень: 3, лучший score: 0.9
[[1, 7, 8], [4, 6, 10], [2, 3, 9]]
Уровень: 4, лучший score: 1.0
[[1, 2, 3], [3, 5, 6, 7], [8, 9], [4, 6, 10]]


In [11]:
best_comb

[1023]